In [1]:
import sys

from pprint import pprint as pp

import requests
import dns.resolver # NOTE: dnspython package
import tldextract

NameError: name 'connection' is not defined

In [18]:

class CustomAdapter(requests.adapters.HTTPAdapter):
    def __init__(self, nameservers):
        self.nameservers = nameservers
        super().__init__()

    def resolve(self, host, nameservers, record_type):
        dns_resolver = dns.resolver.Resolver()
        dns_resolver.nameservers = nameservers
        answers = dns_resolver.resolve(host, record_type)
        for rdata in answers:
            return str(rdata)

    def get_connection(self, url, proxies=None):
        ext = tldextract.extract(url)
        fqdn = ".".join([ ext.subdomain, ext.domain, ext.suffix ])

        print("FQDN: {}".format(fqdn))
        a_record = self.resolve(fqdn, self.nameservers, 'A')
        print("A record: {}".format(a_record))

        resolved_url = url.replace(fqdn, a_record) # NOTE: Replace first occurrence only
        print("Resolved URL: {}".format(resolved_url))

        return super().get_connection(resolved_url, proxies=proxies)

In [3]:
def get_with_dns(url):
    nameservers = [
            '185.51.200.2',
            '178.22.122.100'
        ]

    session = requests.Session()
    session.mount(url, CustomAdapter(nameservers))
    return session.get(url)
def get_without_dns(url):
    session = requests.Session()
    return session.get(url)

In [16]:
res = get_without_dns('https://chromedriver.storage.googleapis.com/LATEST_RELEASE_72.0.3626')
print(res.content)
res = get_without_dns(f'https://chromedriver.storage.googleapis.com/{str(res.content.decode("utf-8"))}/chromedriver_win32.zip')
print(res.text)

b"<?xml version='1.0' encoding='UTF-8'?><Error><Code>AccessDenied</Code><Message>Access denied.</Message><Details>We're sorry, but this service is not available in your location</Details></Error>"
<?xml version='1.0' encoding='UTF-8'?><Error><Code>AccessDenied</Code><Message>Access denied.</Message><Details>We're sorry, but this service is not available in your location</Details></Error>


In [19]:

res = get_with_dns('https://chromedriver.storage.googleapis.com/LATEST_RELEASE_72.0.3626')
print(res.content)
res = get_with_dns(f'https://chromedriver.storage.googleapis.com/{str(res.content.decode("utf-8"))}/chromedriver_win32.zip')
print(res.text)


FQDN: chromedriver.storage.googleapis.com
A record: 46.105.191.85
Resolved URL: https://46.105.191.85/LATEST_RELEASE_72.0.3626


SSLError: HTTPSConnectionPool(host='46.105.191.85', port=443): Max retries exceeded with url: /LATEST_RELEASE_72.0.3626 (Caused by SSLError(SSLError(1, '[SSL: SSLV3_ALERT_HANDSHAKE_FAILURE] sslv3 alert handshake failure (_ssl.c:1108)')))